# South Barnegat Bay Onshore Wind Model Prediction with the Use of Long Short-Term Memory Neural Networks - LSTM Model (Depreciated)

### Written by Krupam Patel
### Written in Python 3.11.14

###### This file contains the code for the LSTM for the wind speeds to be predicted.

In [335]:
from tensorflow import keras # type: ignore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from datetime import datetime
import os
import pandas as pd
import numpy as np

In [336]:
TARGET = "Direction (A)"
WINDOW = 48

In [337]:
#Opens a new dataframe with the Clean csv
cleancsv = pd.read_csv('CLEAN.csv')
cleancsv = cleancsv.sort_values("Date").reset_index(drop=True)

In [338]:
#Convert data into Date time and create date filter
cleancsv["Date"] = (
    pd.to_datetime(cleancsv["Date"], errors="coerce")
    + pd.to_timedelta(cleancsv["Hr"], unit="h")
)

cleancsv = cleancsv.drop(columns="Hr")
cleancsv = cleancsv.sort_values("Date").reset_index(drop=True)

if cleancsv["Date"].isna().any():
    raise ValueError("Fix invalid timestamps before training.")

# Display conflicting timestamps for inspection.
duplicate_mask = cleancsv["Date"].duplicated(keep=False)

if duplicate_mask.any():
    print(cleancsv.loc[duplicate_mask].to_string(index=False))
    raise ValueError(
        "Duplicate timestamps found. Review the rows printed above."
    )
# Confirm that 24 rows really represent 24 consecutive hours.
hour_steps = cleancsv["Date"].diff().iloc[1:]

if cleancsv["Date"].isna().any():
    raise ValueError("Fix invalid timestamps before training.")

if cleancsv["Date"].duplicated().any():
    raise ValueError("Resolve duplicate timestamps before training.")

"""
Use this in future if data set needs specific dates
prediction = data.loc{
    (untouched_csv['Date'] > datetime(x, x, x)) &
    (untouched_csv['Date'] < datetime(x, x, x,))
}
"""

"\nUse this in future if data set needs specific dates\nprediction = data.loc{\n    (untouched_csv['Date'] > datetime(x, x, x)) &\n    (untouched_csv['Date'] < datetime(x, x, x,))\n}\n"

In [339]:

# Show every row sharing a timestamp.
duplicates = cleancsv[
    cleancsv.duplicated(subset="Date", keep=False)
]

print("ROWS WITH DUPLICATE TIMESTAMPS:")
print(duplicates.to_string(index=False))

# Show intervals that are not exactly one hour.
time_check = pd.DataFrame({
    "previous_time": cleancsv["Date"].shift(1),
    "current_time": cleancsv["Date"],
    "difference": cleancsv["Date"].diff()
})

bad_intervals = time_check.iloc[1:]
bad_intervals = bad_intervals[
    bad_intervals["difference"] != pd.Timedelta(hours=1)
]

print("\nIRREGULAR TIME INTERVALS:")
print(bad_intervals.to_string(index=False))

print("\nINVALID TIMESTAMPS:", cleancsv["Date"].isna().sum())

ROWS WITH DUPLICATE TIMESTAMPS:
Empty DataFrame
Columns: [Date, Mainland Air Temp, Humidity (%), Direction (A), Wind Speed (A), Gusting, Atmospheric Pressure (IN), Precipitation Rate, Bay Temp, Salinity, LBI Air Temp, Ocean Temp, Onshore, upwelling_flag]
Index: []

IRREGULAR TIME INTERVALS:
Empty DataFrame
Columns: [previous_time, current_time, difference]
Index: []

INVALID TIMESTAMPS: 0


In [340]:
#Create month colomn and restrict to only summer months
summer_mask = cleancsv["Date"].dt.month.isin([6, 7, 8])
cleancsv = cleancsv[summer_mask].reset_index(drop=True)

In [341]:
# Include historical wind speed among the input features.
feature_cols = [
    column for column in cleancsv.columns
    if column != "Date"
]

dataset = cleancsv[feature_cols].to_numpy(dtype=np.float32)
output_data = cleancsv[[TARGET]].to_numpy(dtype=np.float32)

if not np.isfinite(dataset).all():
    raise ValueError("Input data contain missing or infinite values.")

if not np.isfinite(dataset).all():
    raise ValueError("Input data contain missing or infinite values.")

In [342]:
#Length of training data
n = len(cleancsv)
train_end = int(n * 0.80)
test_start = int(np.ceil(n * 0.90))

n = len(cleancsv)
train_end = int(n * 0.80)
test_start = int(np.ceil(n * 0.90))

In [ ]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()

scaler_x.fit(dataset[:train_end])
scaler_y.fit(output_data[:train_end])

scaledx = scaler_x.transform(dataset)
scaledy = scaler_y.transform(output_data)

# Each window ends one hour before its target.
hourly_step = (
    cleancsv["Date"].diff() == pd.Timedelta(hours=1)
)

# Require 24 consecutive hourly intervals:
# 23 inside the input window and 1 leading to the target.
valid_target = (
    hourly_step
    .rolling(window=WINDOW, min_periods=WINDOW)
    .sum()
    .eq(WINDOW)
)

target_rows = np.flatnonzero(valid_target.to_numpy())

print("Valid forecast windows:", len(target_rows))
print("Excluded windows:", (n - WINDOW) - len(target_rows))

if len(target_rows) == 0:
    raise ValueError("No complete 24-hour forecast windows found.")

X = np.stack([
    scaledx[i - WINDOW:i]
    for i in target_rows
])

# Convert bins 1–16 into class indices 0–15.
y = output_data[target_rows] - 1

train_mask = target_rows < train_end
val_mask = (
    (target_rows >= train_end)
    & (target_rows < test_start)
)
test_mask = target_rows >= test_start

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test = X[test_mask]

test_rows = target_rows[test_mask]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape)

Valid forecast windows: 2136
Excluded windows: 0
Training: (1699, 48, 13) (1699, 1)
Validation: (219, 48, 13) (219, 1)
Testing: (218, 48, 13)


In [344]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()

scaler_x.fit(dataset[:train_end])
scaler_y.fit(output_data[:train_end])

scaledx = scaler_x.transform(dataset)
scaledy = scaler_y.transform(output_data)

# Each window ends one hour before its target.
target_rows = np.arange(WINDOW, n)

X = np.stack([
    scaledx[i - WINDOW:i]
    for i in target_rows
])

y = scaledy[target_rows]

train_mask = target_rows < train_end
val_mask = (
    (target_rows >= train_end)
    & (target_rows < test_start)
)
test_mask = target_rows >= test_start

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test = X[test_mask]

test_rows = target_rows[test_mask]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape)

Training: (1699, 48, 13) (1699, 1)
Validation: (219, 48, 13) (219, 1)
Testing: (218, 48, 13)


In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(42)

model = keras.models.Sequential([
    keras.layers.Input(
        shape=(WINDOW, len(feature_cols))
    ),
    keras.layers.LSTM(32),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="softmax")
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [346]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [347]:
#Train the model

#epochs = # of runs
#batch size = how much data is in each batch
training = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    shuffle=False,
    callbacks=[early_stopping]
)

Epoch 1/100


54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.7902 - val_loss: 0.6757
Epoch 2/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.5602 - val_loss: 0.5667
Epoch 3/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4807 - val_loss: 0.5601
Epoch 4/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4580 - val_loss: 0.5076
Epoch 5/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4205 - val_loss: 0.5099
Epoch 6/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4087 - val_loss: 0.4880
Epoch 7/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4001 - val_loss: 0.4942
Epoch 8/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3834 - val_loss: 0.4813
Epoch 9/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3750 - val_loss: 0.4904
Epoch 10/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3676 - val_loss: 0.4804
Epoch 11/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3672 - val_loss: 0.4846
Epoch 12/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3598 - val_lo

In [348]:
prediction_scaled = model.predict(X_test)

# back to original units
prediction = scaler_y.inverse_transform(prediction_scaled).ravel()


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


In [ ]:
# model outputs are standard like y_train.
prediction_scaled = model.predict(X_test)

# Convert back into of wind-speed units
prediction = scaler_y.inverse_transform(prediction_scaled)

# actaul wind speed at the forecast
y_test_true = output_data[test_rows, 0]

# persustebce =
naive_prediction = output_data[test_rows -1, 0]

lstm_mae = mean_absolute_error(y_test_true, prediction)
naive_mae = mean_absolute_error(y_test_true, naive_prediction)

print(f"LSTM test MAE: {lstm_mae:.3f}")
print(f"Persistence test MAE: {naive_mae:.3f}")
print("MAE uses the same units as the CSV's wind-speed column.")

# Check whether the model captures variation.
print(f"Observed standard deviation: {y_test_true.std():.3f}")
print(f"Predicted standard deviation: {prediction.std():.3f}")

first_target = test_rows[0]

print(
    "First input window:",
    cleancsv["Date"].iloc[first_target - WINDOW],
    "through",
    cleancsv["Date"].iloc[first_target - 1]
)

print(
    "First predicted hour:",
    cleancsv["Date"].iloc[first_target]
)


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
LSTM test MAE: 2.127
Persistence test MAE: 1.683
MAE uses the same units as the CSV's wind-speed column.
Observed standard deviation: 5.388
Predicted standard deviation: 4.218
First input window: 2025-08-20 22:00:00 through 2025-08-22 21:00:00
First predicted hour: 2025-08-22 22:00:00


In [350]:
# Saving to csv
y_test_true = np.asarray(y_test_true).ravel()
prediction = np.asarray(prediction).ravel()
naive_prediction = np.asarray(naive_prediction).ravel()

dates = cleancsv["Date"].iloc[test_rows].to_numpy()

# Confirm that every column has one value per forecast hour.
lengths = {
    "Date": len(dates),
    "observed": len(y_test_true),
    "LSTM": len(prediction),
    "naive": len(naive_prediction)
}

if len(set(lengths.values())) != 1:
    raise ValueError(f"Column lengths do not match: {lengths}")

test_df = pd.DataFrame({
    "Date": dates,
    "observed_wind_speed": y_test_true,
    "predicted_wind_speed_lstm": prediction,
    "predicted_wind_speed_naive": naive_prediction
})

os.makedirs("CSV/predictions", exist_ok=True)

test_df.to_csv(
    "CSV/predictions/wind_speed_lstm_predictions.csv",
    index=False
)

In [351]:
print(prediction_scaled.shape)
print(prediction_scaled[:10].ravel())
print(prediction_scaled.mean(), prediction_scaled.std())

(218, 1)
[-0.18414587 -0.01216037  0.2398982   0.4176716   0.5383424   0.72651595
  0.6740313   0.6766829   0.47564027  0.7135244 ]
-0.5056096 0.9971369
